# A/B modeling with and without income features

Model A uses internal and engineered features while excluding income integration columns. Model B uses the same base features and adds income-related columns.

This notebook trains Logistic Regression and HistGradientBoosting on one shared stratified split, reports ROC AUC, PR AUC, log loss, and F1 at the best threshold, and saves `model_comparison.csv`, `model_comparison_bar.png`, and joblib pipelines in `outputs/`.

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / "src" / "config.py").exists():
    root = root.parent
sys.path.insert(0, str(root))

In [2]:
import pandas as pd

from src.config import TELCO_WITH_FEATURES_PATH
from src.evaluate import compare_models, evaluate_binary_classifier, save_model_comparison
from src.plots import plot_model_comparison_bar
from src.train_models import save_ab_models, train_ab_models

df = pd.read_parquet(TELCO_WITH_FEATURES_PATH)
bundle = train_ab_models(df)
y_test = bundle["y_test"]

results = []
for name, model in bundle["models"].items():
    X_te = bundle["X_test_A"] if name.startswith("A") else bundle["X_test_B"]
    results.append(evaluate_binary_classifier(model, X_te, y_test, name))

comp = compare_models(results)
comp

,roc_auc,pr_auc,log_loss,f1_best_threshold
name,,,,
A_logreg,0.837881,0.637346,0.479666,0.625592
A_hgb,0.844135,0.648702,0.430371,0.633157
B_logreg,0.838063,0.637687,0.479399,0.625940
B_hgb,0.844135,0.648702,0.430371,0.633157


In [3]:
path = save_model_comparison(comp)
print("Saved:", path)

fig_path = plot_model_comparison_bar(comp)
print("Figure:", fig_path)

model_paths = save_ab_models(bundle)
print("Joblib:", model_paths)

Saved: E:\Personal Project\telecom-churn-income-integration\outputs\tables\model_comparison.csv
Figure: E:\Personal Project\telecom-churn-income-integration\outputs\figures\model_comparison_bar.png
Joblib: {'A_logreg': WindowsPath('E:/Personal Project/telecom-churn-income-integration/outputs/models/modelA_logreg.joblib'), 'A_hgb': WindowsPath('E:/Personal Project/telecom-churn-income-integration/outputs/models/modelA_hgb.joblib'), 'B_logreg': WindowsPath('E:/Personal Project/telecom-churn-income-integration/outputs/models/modelB_logreg.joblib'), 'B_hgb': WindowsPath('E:/Personal Project/telecom-churn-income-integration/outputs/models/modelB_hgb.joblib')}
